<a href="https://colab.research.google.com/github/kkiattikulpimol/Khemmanat.github.io/blob/main/1st_Person.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shard 1 (0-25%)

In [ ]:
!pip install -q youtube-transcript-api pandas numpy matplotlib seaborn langdetect tqdm
from google.colab import drive
import os

if not os.path.exists('/content/drive/My Drive'):
    try:
      drive.flush_and_unmount()
    except ValueError:
      pass
    if os.path.exists('/content/drive'):
        !rm -rf /content/drive
    drive.mount('/content/drive', force_remount=True)
    print("Google Drive mounted successfully.")
else:
    print("Google Drive is already mounted.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.1/485.1 kB 19.4 MB/s eta 0:00:00
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
Google Drive mounted successfully.


Pathway

In [ ]:
import os, math, pandas as pd, numpy as np
BASE = "/content/drive/MyDrive/Final EGBI222 Group Project"
CHUNK = f"{BASE}/chunks"; os.makedirs(CHUNK, exist_ok=True)
print(f"Base file folder set to: {BASE}")
print(f"Chunks folder set to: {CHUNK}")

Base file folder set to: /content/drive/MyDrive/Final EGBI222 Group Project
Chunks folder set to: /content/drive/MyDrive/Final EGBI222 Group Project/chunks


Load Dataset

In [ ]:
CSV = f"{BASE}/youtube_data.csv"
df = pd.read_csv(CSV).copy()
if 'video_id_clean' not in df.columns and 'video_id' in df.columns:
    df['video_id_clean'] = df['video_id'].astype(str)
ID = 'video_id_clean' if 'video_id_clean' in df.columns else 'video_id'
df = df.reset_index(drop=True); df['row_no'] = df.index + 1

Quater1 slice

In [ ]:
N = len(df); q1 = math.ceil(N/4)
lo, hi = 1, q1
df_small = df[(df['row_no']>=lo)&(df['row_no']<=hi)].copy().reset_index(drop=True)
print(f"P1 rows {lo}-{hi}/{N} → {len(df_small)}")

P1 rows 1-4398/17589 → 4398


Captions

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from concurrent.futures import ThreadPoolExecutor, as_completed
from langdetect import detect
from tqdm import tqdm

def fetch_cap(vid):
    vid = str(vid)
    try:
        trs = YouTubeTranscriptApi.list_transcripts(vid)
        text = ""
        lang = 'unknown'
        source = 'yt_caption'

        try:
            tr = trs.find_transcript(['en'])
            lang = 'en'
            segs = tr.fetch()
            text = " ".join(s.get('text','') for s in segs if s.get('text'))
        except NoTranscriptFound:
            try:
                tr = next(iter(trs))
                lang = tr.language_code or 'unknown'
                segs = tr.fetch()
                text = " ".join(s.get('text','') for s in segs if s.get('text'))
            except Exception:
                text = ""
                lang = "unknown"
                source = "no_caption"
        except Exception:
             text = ""
             lang = "unknown"
             source = "no_caption"


        lg = 'unknown'
        if text.strip():
            try: lg = detect(text)
            except: lg = lang or 'unknown'
        return vid, text, lg, source
    except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable, Exception):
        return vid, "", "unknown", "no_caption"

vids = df_small[ID].astype(str).tolist()
rows = []
with ThreadPoolExecutor(max_workers=32) as ex:
    futs = {ex.submit(fetch_cap, v): v for v in vids}
    for f in tqdm(as_completed(futs), total=len(futs), desc="P1 captions"):
        rows.append(f.result())

cap = pd.DataFrame(rows, columns=[ID,'transcript','lang_guess','source'])
out = df_small[[ID,'title','description'] if 'title' in df_small.columns or 'description' in df_small.columns else [ID]].merge(cap, on=ID, how='left')

P1 captions: 100%|██████████| 4398/4398 [00:00<00:00, 147347.25it/s]


Metadata fallback (title/description)

In [ ]:
title_col = 'title' if 'title' in df_small.columns else None
desc_col  = 'description' if 'description' in df_small.columns else None
if title_col or desc_col:
    mask = (out['transcript'].isna()) | (out['transcript']=="")
    if title_col and desc_col:
        out.loc[mask,'transcript'] = (df_small.loc[mask, title_col].fillna('') + " " +
                                      df_small.loc[mask, desc_col].fillna('')).str.strip()
    elif title_col:
        out.loc[mask,'transcript'] = df_small.loc[mask, title_col].fillna('')
    elif desc_col:
        out.loc[mask,'transcript'] = df_small.loc[mask, desc_col].fillna('')
    out.loc[mask & (out['lang_guess'].isna() | (out['lang_guess']=="unknown")), 'lang_guess'] = 'unknown'
    out.loc[mask & (out['source'].isna()), 'source'] = 'meta_fallback'

mask_no_caption_no_meta = (out['source'] == 'no_caption') & ((out['transcript'].isna()) | (out['transcript']==""))
out.loc[mask_no_caption_no_meta, 'lang_guess'] = 'unknown'
out.loc[mask_no_caption_no_meta, 'source'] = 'no_caption'

Save shard

In [ ]:
shard = f"{CHUNK}/transcripts_p1_{lo}-{hi}.csv"; out.to_csv(shard, index=False); print("Saved shard:", shard)

Saved shard: /content/drive/MyDrive/Final EGBI222 Group Project/chunks/transcripts_p1_1-4398.csv


Transcript to English

In [ ]:
!pip -q install langdetect==1.0.9 deep-translator==1.11.4 tqdm==4.66.5

from google.colab import drive
from langdetect import detect, DetectorFactory, LangDetectException
from deep_translator import GoogleTranslator, MyMemoryTranslator
from tqdm.auto import tqdm
import pandas as pd, numpy as np
import os, re, time, random, sys

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive is already mounted.")

BASE   = "/content/drive/MyDrive/Final EGBI222 Group Project"
IN_CSV = os.path.join(BASE, "chunks", "transcripts_p1_1-4398.csv")
OUTDIR = os.path.join(BASE, "chunk2")
OUT_CSV = os.path.join(OUTDIR, "transcripts_en_p1_1-4398.csv")
AUDIO_DIR = f"{BASE}/audio"

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
print("Input :", IN_CSV)
print("Output:", OUT_CSV)

try:
    df = pd.read_csv(IN_CSV)
    if 'transcript' not in df.columns:
        for c in ['text', 'caption', 'content']:
            if c in df.columns:
                df = df.rename(columns={c: 'transcript'})
                break
    if 'transcript' not in df.columns:
        raise KeyError("Input must contain a 'transcript' column (or one of ['text','caption','content']).")
    print(f"Loaded rows: {len(df)}")
except Exception as e:
    print(f"[FATAL] Could not load input CSV: {e}")
    raise

def clean_text(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s)
    s = s.replace("\u200b","").replace("\u200c","").replace("\u200d","").replace("\ufeff","")
    s = re.sub(r"[\u0000-\u0008\u000B\u000C\u000E-\u001F\u007F]", "", s)
    s = re.sub(r"[^\S\r\n\t]+", " ", s).strip()
    s = re.sub(r"(.)\1{5,}", r"\1\1\1\1\1", s)
    return s

tqdm.pandas(desc="Cleaning")
df['transcript'] = df['transcript'].progress_apply(clean_text)
df = df[df['transcript'].str.strip() != ""].copy()
print("Rows after cleaning:", len(df))

DetectorFactory.seed = 0

def quick_ascii_english(s: str) -> bool:
    """Fast check: mostly ASCII & contains vowels/spaces -> likely English."""
    if not s: return False
    if not all(ord(ch) < 128 for ch in s):
        return False
    return bool(re.search(r"[aeiouAEIOU]", s))

def detect_lang_safe(s: str) -> str:
    s = (s or "").strip()
    if not s: return "und"
    if quick_ascii_english(s):
        return "en"
    try:
        return detect(s) or "und"
    except LangDetectException:
        return "und"
    except Exception:
        return "und"

if 'lang' not in df.columns:
    tqdm.pandas(desc="Detecting language")
    df['lang'] = df['transcript'].progress_apply(detect_lang_safe)
else:
    mask_unknown = df['lang'].astype(str).str.lower().isin(['unknown','und','','nan']) | df['lang'].isna()
    if mask_unknown.any():
        tqdm.pandas(desc="Re-detecting language")
        df.loc[mask_unknown, 'lang'] = df.loc[mask_unknown, 'transcript'].progress_apply(detect_lang_safe)
    df.loc[df['lang'].astype(str).str.lower().isin(['unknown','','nan']) | df['lang'].isna(), 'lang'] = 'und'

print("Language counts:\n", df['lang'].value_counts(dropna=False))

BATCH_SIZE = 80
MAX_TRIES  = 3
BASE_SLEEP = 0.6

if 'transcript_en' not in df.columns:
    df['transcript_en'] = ""

mask_en = df['lang'].astype(str).str.lower().eq('en')
df.loc[mask_en, 'transcript_en'] = df.loc[mask_en, 'transcript']

def translate_batch_google(texts, src_lang):
    """Batch translate via GoogleTranslator with retries. Returns list[str]."""
    texts = [("" if pd.isna(t) else str(t)) for t in texts]
    src = src_lang if src_lang and src_lang not in ("und","auto") else "auto"
    out = list(texts)
    for attempt in range(1, MAX_TRIES+1):
        try:
            gt = GoogleTranslator(source=src, target='en')
            res = gt.translate_batch(texts)
            if isinstance(res, list) and len(res) == len(texts):
                out = [r if (isinstance(r, str) and r.strip() != "") else orig for r,orig in zip(res, texts)]
                return out
        except Exception as e:
            time.sleep(BASE_SLEEP*(2**(attempt-1)) + random.uniform(0,0.3))
    return out

def translate_item_fallback(text, src_lang):
    """Per-item fallback using Google first, then MyMemory."""
    if not text: return ""
    if quick_ascii_english(text): return text
    src = src_lang if src_lang and src_lang not in ("und","auto") else "auto"

    for attempt in range(1, MAX_TRIES+1):
        try:
            r = GoogleTranslator(source=src, target='en').translate(text)
            if r and r.strip(): return r
        except Exception:
            time.sleep(BASE_SLEEP*(2**(attempt-1)) + random.uniform(0,0.3))

    for attempt in range(1, MAX_TRIES+1):
        try:
            r = MyMemoryTranslator(source=src, target='en').translate(text)
            if r and r.strip(): return r
        except Exception:
            time.sleep(BASE_SLEEP*(2**(attempt-1)) + random.uniform(0,0.3))

    return text

lang_order = df['lang'].value_counts().index.tolist()
lang_order = [l for l in lang_order if str(l).lower() not in ('en','und')] + (['und'] if 'und' in lang_order else [])

for lc in lang_order:
    mask = (df['lang'] == lc) & (df['transcript'].str.strip() != "") & (df['transcript_en'].str.strip() == "")
    idxs = df.index[mask].tolist()
    if not idxs:
        continue

    texts = df.loc[idxs, 'transcript'].tolist()
    translated = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"{lc}->en (batch)"):
        chunk = texts[i:i+BATCH_SIZE]
        out = translate_batch_google(chunk, lc)

        fixed = []
        for orig, got in zip(chunk, out):
            if (not got.strip()) or (got.strip() == orig.strip() and not quick_ascii_english(orig)):
                fixed.append(translate_item_fallback(orig, lc))
            else:
                fixed.append(got)
        translated.extend(fixed)

    df.loc[idxs, 'transcript_en'] = translated

need_fix = (df['transcript_en'].str.strip() == "")
if need_fix.any():
    print(f"Rescue pass on {need_fix.sum()} rows…")
    for idx in tqdm(df.index[need_fix], desc="rescue"):
        src = df.at[idx, 'lang']
        txt = df.at[idx, 'transcript']
        df.at[idx, 'transcript_en'] = translate_item_fallback(txt, src)

df['transcript_en'] = df['transcript_en'].fillna("")

if 'lang' in df.columns:
    df = df.drop(columns=['lang'])

try:
    df.to_csv(OUT_CSV, index=False, encoding='utf-8')
    print("✅ Saved:", OUT_CSV)
except Exception as e:
    print(f"❌ Save error: {e}")
    raise

cols = ['video_id_clean' if 'video_id_clean' in df.columns else ('video_id' if 'video_id' in df.columns else df.columns[0]),
        'transcript','transcript_en']
print("\nHead:")
print(df[cols].head(3))
print("\nDone.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.5 which is incompatible.
Google Drive is already mounted.
Input : /content/drive/MyDrive/Final EGBI222 Group Project/chunks/transcripts_p1_1-4398.csv
Output: /content/drive/MyDrive/Final EGBI222 Group Project/chunk2/transcripts_en_p1_1-4398.csv
Loaded rows: 4398


Cleaning:   0%|          | 0/4398 [00:00<?, ?it/s]

Rows after cleaning: 4397


Detecting language:   0%|          | 0/4397 [00:00<?, ?it/s]

Language counts:
 lang
en       3276
es        170
pt        148
ar         80
ja         80
ru         77
fr         76
ko         68
de         59
pl         37
und        34
it         31
th         27
vi         24
tr         21
cs         18
bg         14
hu         13
sv         11
ca         11
no         11
he         11
sk          9
sl          8
el          8
so          8
id          7
zh-tw       7
da          6
nl          6
mk          4
fa          4
zh-cn       4
et          4
ro          4
sw          4
uk          4
fi          3
hr          3
cy          2
tl          2
lt          1
bn          1
sq          1
Name: count, dtype: int64


...

lt->en (batch):   0%|          | 0/1 [00:00<?, ?it/s]

bn->en (batch):   0%|          | 0/1 [00:00<?, ?it/s]

sq->en (batch):   0%|          | 0/1 [00:00<?, ?it/s]

und->en (batch):   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved: /content/drive/MyDrive/Final EGBI222 Group Project/chunk2/transcripts_en_p1_1-4398.csv

Head:
  video_id_clean                                         transcript  \
0    --F7dc-_FSI  «السودان ينتفض» أمام السفارة بالقاهرة حرية سلا...   
1    --cCAD-8Y_U  Pokemon Tower Defense Episodio 2 Espero que te...   
2    --g2gG8pQ0w  New Hip Hop - Kemo Treats - Pancakes Download ...   

                                       transcript_en  
0  “Sudan rises up” in front of the embassy in Ca...  
1  Pokemon Tower Defense Episodio 2 Espero que te...  
2  New Hip Hop - Kemo Treats - Pancakes Download ...  

Done.
